# 🎵 MusicScope™ - Complete Analytics Dashboard

**Date**: 2025-09-16  
**Version**: Enhanced 4.0  
**Brand**: MusicScope™  

## 🚀 **Complete Feature Set**:
- 📊 **ChartFlow™**: Interactive performance charts & artist comparison
- 🎭 **SentimentScope™**: Fan insights, comment analysis & tour planning
- 🎬 **ContentFlow™**: Video categorization & content strategy
- 🤖 **Auto-Generated Summaries**: Intelligent insights with actionable recommendations
- 💝 **Compassionate Analytics**: Treats artists as humans, not data points

## 📋 **Navigation**:
1. [Setup & Data Generation](#setup)
2. [ChartFlow™ - Performance Analytics](#chartflow)
3. [SentimentScope™ - Fan Insights](#sentimentscope)
4. [ContentFlow™ - Video Analysis](#contentflow)
5. [Auto-Generated Summary](#summary)

**One comprehensive notebook with all analytics features!** 🎯

---
# 🔧 Setup & Data Generation {#setup}

Loading all analytics modules and generating sample data for demonstration.

In [ ]:
# 🎵 MusicScope™ - Complete Analytics Toolkit
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go

# Import all our custom modules
from youtubeviz.storytelling import story_block, quick_takeaways, narrative_intro
from youtubeviz.charts import (
    views_over_time_plotly, enhance_chart_beauty,
    create_divergent_sentiment_chart, create_sentiment_cluster_chart
)
from youtubeviz.sentiment import (
    extract_top_positive_comments, extract_top_negative_comments_with_percentages,
    identify_standout_videos, analyze_roster_sentiment
)
from youtubeviz.config_validation import (
    get_artists_from_env, validate_artist_count_in_data, 
    print_validation_results, EXPECTED_ARTIST_COUNT, EXPECTED_ARTISTS
)

print('🎯 MusicScope™ Complete Analytics Toolkit Loaded!')
print('✅ ChartFlow™: Performance charts ready')
print('✅ SentimentScope™: Fan insights ready')
print('✅ ContentFlow™: Video analysis ready')
print(f'🎵 Expected Artists: {EXPECTED_ARTIST_COUNT} ({', '.join(EXPECTED_ARTISTS)})')

In [ ]:
# 🎯 ARTIST VALIDATION: Automatically get artists from .env configuration
artists, artist_count = get_artists_from_env()
print(f'📊 Generating sample data for {artist_count} artists: {', '.join(artists)}')

# Validate we have the expected number of artists
if artist_count != EXPECTED_ARTIST_COUNT:
    print(f'🚨 WARNING: Expected {EXPECTED_ARTIST_COUNT} artists, found {artist_count}')
    print('Check your .env file configuration!')

dates = pd.date_range(end=datetime.now(), periods=30, freq='D')

# Performance data with realistic differences to identify clear winners
# Each artist gets different performance characteristics
artist_profiles = {
    'Flyana Boss': {'base_views': 85000, 'growth_rate': 0.03, 'volatility': 0.2},  # Strong performer, growing
    'BiC Fizzle': {'base_views': 65000, 'growth_rate': 0.02, 'volatility': 0.15}, # Solid, steady growth
    'COBRAH': {'base_views': 45000, 'growth_rate': 0.05, 'volatility': 0.25},     # Smaller but fastest growth
    'Raiche': {'base_views': 55000, 'growth_rate': 0.01, 'volatility': 0.18},    # Stable, slow growth
    're6ce': {'base_views': 35000, 'growth_rate': -0.01, 'volatility': 0.3},     # Declining, needs attention
    'Corook': {'base_views': 75000, 'growth_rate': 0.025, 'volatility': 0.12}   # High performer, consistent
}

performance_data = []
for i, date in enumerate(dates):
    for artist in artists:
        profile = artist_profiles[artist]
        
        # Calculate trending views based on growth rate and time
        trend_multiplier = 1 + (profile['growth_rate'] * i)
        base_views = profile['base_views'] * trend_multiplier
        
        # Add realistic daily variation
        daily_variation = np.random.normal(1, profile['volatility'])
        views = max(1000, int(base_views * daily_variation))
        
        likes = int(views * np.random.uniform(0.03, 0.07))
        comments = int(views * np.random.uniform(0.008, 0.018))
        
        performance_data.append({
            'artist_name': artist,
            'date': date,
            'views': views,
            'likes': likes,
            'comments': comments,
            'engagement_rate': (likes + comments) / views * 100
        })

df = pd.DataFrame(performance_data)
print(f'📊 Generated {len(df)} performance records for {len(artists)} artists')
print(f'📅 Date range: {df.date.min().date()} to {df.date.max().date()}')
print(f'🎤 Artists: {", ".join(artists)}')
print(f'🔍 Unique artists in data: {df.artist_name.nunique()}')

# 🚨 LOUD VALIDATION: Check artist count matches .env configuration
validation_result = validate_artist_count_in_data(df, 'artist_name')
print_validation_results(validation_result, loud=True)

# Stop execution if validation fails
if not validation_result['valid']:
    raise ValueError('Artist validation failed! Check .env configuration or run ETL pipeline.')

df.head()

---
# 📊 ChartFlow™ - Performance Analytics {#chartflow}

Interactive charts showing artist performance, engagement metrics, and growth trends.

In [ ]:
# 🗄️ DATABASE VALIDATION: Check if database has correct artist count
print('🔍 Validating database artist count...')
try:
    from youtubeviz.config_validation import validate_database_artist_count
    db_validation = validate_database_artist_count()
    print_validation_results(db_validation, loud=True)
    
    if not db_validation['valid']:
        print('\n⚠️  Database validation failed, but continuing with sample data for demo.')
        print('💡 To fix: Run ETL pipeline with: python tools/etl/run_focused_etl.py')
    else:
        print('✅ Database validation passed! Real data is available.')
        
except Exception as e:
    print(f'⚠️  Database connection failed: {e}')
    print('💡 Continuing with sample data for demo purposes.')

In [ ]:
# ChartFlow™ - Daily Views Analysis with Momentum Trends
# Aggregate views by day per artist to show true performance trends
daily_views = df.groupby(['artist_name', 'date'])['views'].sum().reset_index()
daily_views = daily_views.sort_values(['artist_name', 'date'])

# Calculate 7-day rolling average to show momentum
daily_views['rolling_avg'] = daily_views.groupby('artist_name')['views'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

# Create enhanced interactive chart
fig = px.line(
    daily_views,
    x='date',
    y='views',
    color='artist_name',
    title='📈 ChartFlow™ - Daily YouTube Views with Momentum Trends',
    hover_data={'rolling_avg': ':.0f'}
)

# Add rolling average traces for momentum visualization
for artist in daily_views['artist_name'].unique():
    artist_data = daily_views[daily_views['artist_name'] == artist]
    fig.add_trace(go.Scatter(
        x=artist_data['date'],
        y=artist_data['rolling_avg'],
        mode='lines',
        name=f'{artist} (7-day avg)',
        line=dict(dash='dash', width=2),
        opacity=0.7,
        showlegend=False
    ))

# Enhanced interactivity
fig.update_layout(
    hovermode='x unified',
    height=500,
    xaxis_title='Date',
    yaxis_title='Daily Views',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

# Calculate actual performance metrics
total_days = len(daily_views['date'].unique())
total_artists = daily_views['artist_name'].nunique()
avg_daily_views = daily_views.groupby('artist_name')['views'].mean()
top_performer = avg_daily_views.idxmax()
top_views = avg_daily_views.max()

story_block(
    fig=fig,
    title='🎯 Daily Performance Trends & Momentum Analysis',
    bullets=[
        f'Tracking {total_artists} artists over {total_days} days with daily view aggregation',
        f'Top performer: {top_performer} averaging {top_views:,.0f} daily views',
        'Solid lines show daily views, dashed lines show 7-day momentum trends',
        'Interactive: hover to see exact values, click legend to focus on specific artists',
        'Look for consistent upward trends (momentum) vs. one-time spikes'
    ],
    caption='ChartFlow™ - Enhanced performance visualization with momentum analysis'
)